# RF



## Data Set 1

In [15]:
# INSTALLING **PACKAGES**


!pip install statsmodels

! pip install optuna

!pip install tabulate

# Standard libraries
import sys  # System-specific parameters and functions
import os   # Miscellaneous operating system interfaces
import warnings  # Warning control
warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd  # Data manipulation and analysis
import numpy as np  # Numerical operations

# Visualization
import matplotlib.pyplot as plt  # Plotting library
import seaborn as sns  # Statistical data visualization
from matplotlib.colors import ListedColormap  # Colormap utilities

# Model Helpers
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler  # Preprocessing tools
from sklearn import model_selection, metrics, preprocessing  # Model selection, evaluation, and preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV  # Model selection and evaluation
from sklearn.tree import DecisionTreeClassifier  # Decision tree classifier


# Statistical analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor  # Variance inflation factor
from scipy.stats import pointbiserialr, chi2_contingency, spearmanr, entropy  # Statistical functions
from statsmodels.graphics.gofplots import qqplot  # Q-Q plot
from collections import Counter  # Container datatypes

# Tabulate
from tabulate import tabulate  # Pretty-print tabular data

# Set visualization style
#sns.set()  # Set Seaborn default style
#plt.style.use('ggplot')  # Set ggplot style for matplotlib



# LOADING THE DATA SET
#https://drive.google.com/file/d/1Z_KsoIumw-fvivVombIoWuRo0LOe2nCb/view?usp=sharing
#https://drive.google.com/file/d/1aD1PXfwEEZ_F2lQgxuPfj-TVbxQ6NajK/view?usp=sharing
import gdown
import pandas as pd



# Read Xlsx file
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import pandas as pd
import numpy as np  # Ensure numpy is imported
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Read Xlsx file
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import pandas as pd
import numpy as np  # Ensure numpy is imported
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import pandas as pd


import pandas as pd

# Updated Google Sheets link setup
sheet_id = "1j_Euo80PrGckVDVr2hTG9zZebxJD0TSC"
sheet_name = "Sheet1"  # Update this if the sheet name is different

# Generate the CSV export link
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

# Read the Google Sheet as CSV
df = pd.read_csv(csv_url)

# Backup original DataFrame
df_backup = df.copy()

df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date').sort_index()


# Display the DataFrame (optional in scripts)
df.head()
df_backup=df.copy()

df = df.drop(columns=["YEAR", "MO", "DY"])

df
df.columns
df=df[["WS10M"]]
df.columns
df

# Define the number of lag terms
num_lags = 3

# Create lag features for each column in the dataset
lagged_df = df.copy()
for lag in range(1, num_lags + 1):
    lagged_features = df.shift(lag).add_suffix(f'_lag{lag}')
    lagged_df = pd.concat([lagged_df, lagged_features], axis=1)

# Drop rows with NaN values (since initial lags will have missing values)
lagged_df = lagged_df.dropna()



# Display the first few rows
print(lagged_df.head())
lagged_df

df = lagged_df
df
## Data Sets

# 1


df
###########################################################################################
# Define the list of variables to select

X = df.drop(columns=['WS10M'])
# Select only those columns from X
 # Features (lagged variables)
y = df['WS10M']  # Target variable (wind speed)

# Define the split index
split_index = int(len(X) * 0.8)  # 80% for training, 20% for testing

# Training set
X_train = X[:split_index]
y_train = y[:split_index]

# Testing set
X_test = X[split_index:]
y_test = y[split_index:]

numerical_cols=df.columns
numerical_cols

print("X_train:")
print(X_train.head())
print("y_train:")
print(y_train.head())
print("X_test:")
print(X_test.head())
print("y_test:")
print(y_test.head())

# Print the shapes of the datasets
print("\nShapes:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


            WS10M  WS10M_lag1  WS10M_lag2  WS10M_lag3
Date                                                 
2013-01-04   5.74        4.75        4.25        3.99
2013-01-05   5.79        5.74        4.75        4.25
2013-01-06   4.52        5.79        5.74        4.75
2013-01-07   4.66        4.52        5.79        5.74
2013-01-08   3.61        4.66        4.52        5.79
X_train:
            WS10M_lag1  WS10M_lag2  WS10M_lag3
Date                                          
2013-01-04        4.75        4.25        3.99
2013-01-05        5.74        4.75        4.25
2013-01-06        5.79        5.74        4.75
2013-01-07        4.52        5.79        5.74
2013-01-08        4.66        4.52        5.79
y_train:
Date
2013-01-04    5.74
2013-01-05    5.79
2013-01-06    4.52
2013-01-07    4.66
2013-01-08    3.61
Name: WS10M, dtype: float64
X_test:
            WS10M_lag1  WS10M_lag2  WS10M_lag3
Date                                          
2021-10-20        3.92        6.13        6.8

### Optuna

In [16]:
!pip install optuna

In [17]:
# ============================================================
# RANDOM FOREST + OPTUNA (TIME-SERIES SAFE, FULL CODE)
# ============================================================

import optuna
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# Time Series Cross-Validation
# -----------------------------
tscv = TimeSeriesSplit(n_splits=5)

# -----------------------------
# Optuna Objective Function
# (TRAINING SET ONLY)
# -----------------------------
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 50, 200, step=50)
    max_depth = trial.suggest_categorical('max_depth', [None, 10, 20, 30])
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 7)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    fold_mse = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        y_val_pred = model.predict(X_val)

        mse = mean_squared_error(y_val, y_val_pred)
        fold_mse.append(mse)

    return np.mean(fold_mse)

# -----------------------------
# Run Optuna
# -----------------------------
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=50)

best_params = study.best_params
print(f"\n✅ Best Parameters Found by Optuna:\n{best_params}")

# -----------------------------
# Train Final Model
# -----------------------------
best_rf_optuna = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_rf_optuna.fit(X_train, y_train)

# Predictions
y_train_pred_optuna = best_rf_optuna.predict(X_train)
y_test_pred_optuna = best_rf_optuna.predict(X_test)

# -----------------------------
# Evaluation Metrics
# -----------------------------
train_mse_optuna = mean_squared_error(y_train, y_train_pred_optuna)
test_mse_optuna = mean_squared_error(y_test, y_test_pred_optuna)

train_mae_optuna = mean_absolute_error(y_train, y_train_pred_optuna)
test_mae_optuna = mean_absolute_error(y_test, y_test_pred_optuna)

train_rmse_optuna = np.sqrt(train_mse_optuna)
test_rmse_optuna = np.sqrt(test_mse_optuna)

train_r2_optuna = r2_score(y_train, y_train_pred_optuna)
test_r2_optuna = r2_score(y_test, y_test_pred_optuna)

train_mape_optuna = np.mean(
    np.abs((y_train - y_train_pred_optuna) / np.maximum(y_train, 1e-8))
) * 100

test_mape_optuna = np.mean(
    np.abs((y_test - y_test_pred_optuna) / np.maximum(y_test, 1e-8))
) * 100

print("\n📊 Evaluation Metrics:")
print(f"Train MSE: {train_mse_optuna:.4f}, Test MSE: {test_mse_optuna:.4f}")
print(f"Train MAE: {train_mae_optuna:.4f}, Test MAE: {test_mae_optuna:.4f}")
print(f"Train RMSE: {train_rmse_optuna:.4f}, Test RMSE: {test_rmse_optuna:.4f}")
print(f"Train MAPE: {train_mape_optuna:.2f}%, Test MAPE: {test_mape_optuna:.2f}%")
print(f"Train R²: {train_r2_optuna:.4f}, Test R²: {test_r2_optuna:.4f}")

# -----------------------------
# Optuna Results Table
# -----------------------------
optuna_results = study.trials_dataframe()

cols_to_extract = [
    'number', 'value',
    'params_n_estimators',
    'params_max_depth',
    'params_min_samples_split',
    'params_min_samples_leaf',
    'params_max_features'
]

optuna_table_df = optuna_results[cols_to_extract]

optuna_table_df.columns = [
    'Trial', 'CV MSE', 'n_estimators', 'max_depth',
    'min_samples_split', 'min_samples_leaf', 'max_features'
]

fig_table_optuna = go.Figure(data=[go.Table(
    header=dict(values=list(optuna_table_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[optuna_table_df[col] for col in optuna_table_df.columns],
               fill_color='lavender',
               align='left'))
])

fig_table_optuna.update_layout(
    title="Optuna Results - Random Forest Hyperparameter Tuning",
    height=600,
    template='plotly_white'
)

fig_table_optuna.show()

# -----------------------------
# Time Series Plot
# -----------------------------
train_dates = X_train.index
test_dates = X_test.index

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train_dates, y=y_train,
    mode='lines', name='Y Train'
))

fig.add_trace(go.Scatter(
    x=test_dates, y=y_test,
    mode='lines', name='Y Test'
))

fig.add_trace(go.Scatter(
    x=train_dates, y=y_train_pred_optuna,
    mode='lines', name='Y Train Predicted',
    line=dict(dash='dot')
))

fig.add_trace(go.Scatter(
    x=test_dates, y=y_test_pred_optuna,
    mode='lines', name='Y Test Predicted',
    line=dict(dash='dot')
))

fig.update_layout(
    title="Actual vs Predicted Values (Random Forest + Optuna) Optuna Hyper Parameter tuned",
    xaxis_title="Date",
    yaxis_title="Target Variable",
    template="plotly_white"
)

fig.show()


[I 2026-02-09 05:35:10,391] A new study created in memory with name: no-name-4fa78262-adf6-4f7d-a80c-29a7428a3ebf
[I 2026-02-09 05:35:12,284] Trial 0 finished with value: 0.7025227125826211 and parameters: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7025227125826211.
[I 2026-02-09 05:35:13,475] Trial 1 finished with value: 0.7034847039352414 and parameters: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7025227125826211.
[I 2026-02-09 05:35:17,872] Trial 2 finished with value: 0.6927651873184679 and parameters: {'n_estimators': 150, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 2 with value: 0.6927651873184679.
[I 2026-02-09 05:35:22,330] Trial 3 finished with value: 0.6931422331426169 and parameters: {'n_estimators': 150, 'max_depth': 30,


✅ Best Parameters Found by Optuna:
{'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt'}

📊 Evaluation Metrics:
Train MSE: 0.4315, Test MSE: 0.7749
Train MAE: 0.5039, Test MAE: 0.6762
Train RMSE: 0.6569, Test RMSE: 0.8803
Train MAPE: 14.69%, Test MAPE: 18.95%
Train R²: 0.8288, Test R²: 0.6786


### Original

In [18]:
# ===============================
# RANDOM FOREST WITH TIME-SERIES CV
# ===============================

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------
# 1. Define your dataset
# -------------------------------
# Example:
# df = pd.read_csv("your_data.csv")
# X = df.drop("target_column", axis=1)
# y = df["target_column"]

# Ensure X and y are defined
assert 'X' in globals() and 'y' in globals(), "Define X (features) and y (target) before running."

# -------------------------------
# 2. Split into train + final test
# -------------------------------
train_size = int(len(X) * 0.8)  # 80% training, 20% final test
X_train_full, X_test_final = X.iloc[:train_size], X.iloc[train_size:]
y_train_full, y_test_final = y.iloc[:train_size], y.iloc[train_size:]

# -------------------------------
# 3. Initialize model & CV
# -------------------------------
rf = RandomForestRegressor(n_estimators=100, random_state=42)
tscv = TimeSeriesSplit(n_splits=5)

# Dictionaries to store metrics
metrics_train = {"MSE": [], "MAE": [], "RMSE": [], "MAPE": [], "R2": []}
metrics_val = {"MSE": [], "MAE": [], "RMSE": [], "MAPE": [], "R2": []}

# -------------------------------
# 4. Time-Series Cross-Validation
# -------------------------------
for train_idx, val_idx in tscv.split(X_train_full):
    X_train, X_val = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
    y_train, y_val = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]

    # Train
    rf.fit(X_train, y_train)

    # Predict
    y_train_pred = rf.predict(X_train)
    y_val_pred = rf.predict(X_val)

    # Compute metrics
    for (true, pred, storage) in [(y_train, y_train_pred, metrics_train), (y_val, y_val_pred, metrics_val)]:
        mse = mean_squared_error(true, pred)
        mae = mean_absolute_error(true, pred)
        rmse = np.sqrt(mse)
        mape = np.mean(np.abs((true - pred) / np.maximum(true, 1e-8))) * 100  # positive targets only
        r2 = r2_score(true, pred)

        storage["MSE"].append(mse)
        storage["MAE"].append(mae)
        storage["RMSE"].append(rmse)
        storage["MAPE"].append(mape)
        storage["R2"].append(r2)

# -------------------------------
# 5. Average metrics over CV folds
# -------------------------------
avg_train_metrics = {k: np.mean(v) for k, v in metrics_train.items()}
avg_val_metrics = {k: np.mean(v) for k, v in metrics_val.items()}

# -------------------------------
# 6. Train on full training data and evaluate on final test set
# -------------------------------
rf.fit(X_train_full, y_train_full)
y_test_pred = rf.predict(X_test_final)

final_test_metrics = {
    "MSE": mean_squared_error(y_test_final, y_test_pred),
    "MAE": mean_absolute_error(y_test_final, y_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_final, y_test_pred)),
    "MAPE": np.mean(np.abs((y_test_final - y_test_pred) / np.maximum(y_test_final, 1e-8))) * 100,
    "R2": r2_score(y_test_final, y_test_pred)
}

# -------------------------------
# 7. Print results
# -------------------------------
print("\n📊 Cross-Validated Average Metrics (Training Folds):")
for k, v in avg_train_metrics.items():
    print(f"Train {k}: {v:.4f}")

print("\n📊 Cross-Validated Average Metrics (Validation Folds):")
for k, v in avg_val_metrics.items():
    print(f"Validation {k}: {v:.4f}")

print("\n📊 Final Test Set Metrics:")
for k, v in final_test_metrics.items():
    print(f"Test {k}: {v:.4f}" if k != "MAPE" else f"Test {k}: {v:.2f}%")



📊 Cross-Validated Average Metrics (Training Folds):
Train MSE: 0.0918
Train MAE: 0.2320
Train RMSE: 0.3028
Train MAPE: 6.7984
Train R2: 0.9624

📊 Cross-Validated Average Metrics (Validation Folds):
Validation MSE: 0.7405
Validation MAE: 0.6536
Validation RMSE: 0.8597
Validation MAPE: 18.8162
Validation R2: 0.6981

📊 Final Test Set Metrics:
Test MSE: 0.8290
Test MAE: 0.7006
Test RMSE: 0.9105
Test MAPE: 19.57%
Test R2: 0.6562


In [19]:
import plotly.graph_objects as go

# -------------------------------
# 1. Predictions from the final trained model
# -------------------------------
y_train_pred_full = rf.predict(X_train_full)
y_test_pred_final = rf.predict(X_test_final)

# -------------------------------
# 2. Dates for x-axis
# -------------------------------
train_dates = X_train_full.index
test_dates = X_test_final.index

# -------------------------------
# 3. Create the time series plot
# -------------------------------
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(x=train_dates, y=y_train_full, mode='lines', name='Y Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_dates, y=y_test_final, mode='lines', name='Y Test', line=dict(color='green')))

# Predicted values
fig.add_trace(go.Scatter(x=train_dates, y=y_train_pred_full, mode='lines', name='Y Train Predicted', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=test_dates, y=y_test_pred_final, mode='lines', name='Y Test Predicted', line=dict(color='red', dash='dot')))

# Layout
fig.update_layout(
    title="Actual vs Predicted Values for Training and Test Sets Without Hyperparameter Tuning",
    xaxis_title="Date",
    yaxis_title="Target Variable",
    template="plotly_white"
)

# Show the plot
fig.show()


### Grid Search CV HYper Parameter tuning

In [20]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Define parameter grid (bootstrap removed)
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]  # Valid options only
}

# Initialize RandomForestRegressor
rf = RandomForestRegressor(random_state=42)

# TimeSeriesSplit ensures training precedes testing in time
tscv = TimeSeriesSplit(n_splits=5)

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2,
    refit=True
)

# Fit the model with GridSearchCV
grid_search.fit(X_train, y_train)

# Extract best parameters and model
best_params = grid_search.best_params_
best_score = grid_search.best_score_
best_rf_grid = grid_search.best_estimator_

# Predict with best model
y_train_pred_grid = best_rf_grid.predict(X_train)
y_test_pred_grid = best_rf_grid.predict(X_test)

from sklearn.metrics import r2_score

# Evaluate performance
train_mse = mean_squared_error(y_train, y_train_pred_grid)
test_mse = mean_squared_error(y_test, y_test_pred_grid)

train_mae = mean_absolute_error(y_train, y_train_pred_grid)
test_mae = mean_absolute_error(y_test, y_test_pred_grid)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_r2 = r2_score(y_train, y_train_pred_grid)
test_r2 = r2_score(y_test, y_test_pred_grid)

# Handle division by zero in MAPE
train_mape = np.mean(np.abs((y_train - y_train_pred_grid) / np.maximum(y_train, 1e-8))) * 100
test_mape = np.mean(np.abs((y_test - y_test_pred_grid) / np.maximum(y_test, 1e-8))) * 100

# Output results
print(f"✅ Best Hyperparameters Grid : {best_params}")

print(f"📉 Best CV Score (Negative MSE): {best_score:.4f}")

print(f"Train MSE Grid : {train_mse:.4f}, Test MSE Grid : {test_mse:.4f}")
print(f"Train MAE Grid: {train_mae:.4f}, Test MAE Grid : {test_mae:.4f}")
print(f"Train RMSE Grid : {train_rmse:.4f}, Test RMSE Grid : {test_rmse:.4f}")
print(f"Train MAPE Grid : {train_mape:.2f}%, Test MAPE Grid : {test_mape:.2f}%")
print(f"Train R² Grid : {train_r2:.4f}, Test R² Grid : {test_r2:.4f}")



Fitting 5 folds for each of 243 candidates, totalling 1215 fits
✅ Best Hyperparameters Grid : {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 150}
📉 Best CV Score (Negative MSE): -0.6553
Train MSE Grid : 0.4056, Test MSE Grid : 0.7821
Train MAE Grid: 0.4861, Test MAE Grid : 0.6807
Train RMSE Grid : 0.6369, Test RMSE Grid : 0.8844
Train MAPE Grid : 14.52%, Test MAPE Grid : 19.01%
Train R² Grid : 0.8373, Test R² Grid : 0.6757


In [21]:
# Use integer indices instead of .index
train_dates = np.arange(len(y_train))
test_dates = np.arange(len(y_train), len(y_train) + len(y_test))
# Use integer indices instead of .index
train_dates = np.arange(len(y_train))
test_dates = np.arange(len(y_train), len(y_train) + len(y_test))


In [22]:
import plotly.graph_objects as go

fig = go.Figure()

# Actual train
fig.add_trace(go.Scatter(x=train_dates, y=y_train, mode='lines', name='Y Train', line=dict(color='blue')))

# Actual test
fig.add_trace(go.Scatter(x=test_dates, y=y_test, mode='lines', name='Y Test', line=dict(color='green')))

# Predicted train
fig.add_trace(go.Scatter(x=train_dates, y=y_train_pred_grid, mode='lines', name='Y Train Predicted', line=dict(color='orange', dash='dot')))

# Predicted test
fig.add_trace(go.Scatter(x=test_dates, y=y_test_pred_grid, mode='lines', name='Y Test Predicted', line=dict(color='red', dash='dot')))

fig.update_layout(
    title="Actual vs Predicted Values for Training and Test Sets (Grid Search Tuned)",
    xaxis_title="Time",
    yaxis_title="Target",
    template="plotly_white"
)

fig.show()


### Random Search CV

In [23]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Define the Random Forest Regressor model with fixed random seed
rf = RandomForestRegressor(random_state=42)

# Define the parameter grid for hyperparameter tuning
param_grid = {

    'n_estimators': [50, 100, 150, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 7, 10],
    'min_samples_leaf': [1, 2, 4, 7],
    'max_features': ['auto', 'sqrt', 'log2'],
            # Whether bootstrap samples are used when building trees
}

# Set up the TimeSeriesSplit object for time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Set up RandomizedSearchCV with random_state and TimeSeriesSplit
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=100,
    cv=tscv,
    n_jobs=-1,
    verbose=2,
    scoring='neg_mean_squared_error',
    random_state=42,       # Fixed random seed here
    refit=True
)

# Fit the model
random_search.fit(X_train, y_train)

# Get the best parameters and best estimator
random_best_params = random_search.best_params_
random_best_rf = random_search.best_estimator_

print(f"Best Parameters Random: {random_best_params}")

# Make predictions on train and test sets
y_train_pred_random = random_best_rf.predict(X_train)
y_test_pred_random = random_best_rf.predict(X_test)

from sklearn.metrics import r2_score

# Evaluate performance
random_train_mse = mean_squared_error(y_train, y_train_pred_random)
random_test_mse = mean_squared_error(y_test, y_test_pred_random)

random_train_mae = mean_absolute_error(y_train, y_train_pred_random)
random_test_mae = mean_absolute_error(y_test, y_test_pred_random)

random_train_rmse = np.sqrt(random_train_mse)
random_test_rmse = np.sqrt(random_test_mse)

random_train_mape = np.mean(np.abs((y_train - y_train_pred_random) / np.maximum(y_train, 1e-8))) * 100
random_test_mape = np.mean(np.abs((y_test - y_test_pred_random) / np.maximum(y_test, 1e-8))) * 100

# R-squared
random_train_r2 = r2_score(y_train, y_train_pred_random)
random_test_r2 = r2_score(y_test, y_test_pred_random)

# Print metrics
print(f"Random Train MSE: {random_train_mse:.4f}, Random Test MSE: {random_test_mse:.4f}")
print(f"Random Train MAE: {random_train_mae:.4f}, Random Test MAE: {random_test_mae:.4f}")
print(f"Random Train RMSE: {random_train_rmse:.4f}, Random Test RMSE: {random_test_rmse:.4f}")
print(f"Random Train MAPE: {random_train_mape:.2f}%, Random Test MAPE: {random_test_mape:.2f}%")
print(f"Random Train R²: {random_train_r2:.4f}, Random Test R²: {random_test_r2:.4f}")

# Create a DataFrame from the random search results
random_search_results = pd.DataFrame(random_search.cv_results_)



Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters Random: {'n_estimators': 150, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 10}
Random Train MSE: 0.4056, Random Test MSE: 0.7821
Random Train MAE: 0.4861, Random Test MAE: 0.6807
Random Train RMSE: 0.6369, Random Test RMSE: 0.8844
Random Train MAPE: 14.52%, Random Test MAPE: 19.01%
Random Train R²: 0.8373, Random Test R²: 0.6757


ting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters: {'n_estimators': 100, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 10, 'bootstrap': True}
Random Train MSE: 0.8575, Random Test MSE: 1.2427
Random Train MAE: 0.7003, Random Test MAE: 0.8626
Random Train RMSE: 0.9260, Random Test RMSE: 1.1147
Random Train MAPE: 17.3376%, Random Test MAPE: 19.1552%

In [32]:
full_dates = np.arange(len(y_train) + len(y_test))
full_actual = np.concatenate([y_train, y_test])
full_pred = np.concatenate([y_train_pred, y_test_pred])


In [33]:
import plotly.graph_objects as go

fig = go.Figure()

# Plot actual full series
fig.add_trace(go.Scatter(
    x=full_dates, y=full_actual,
    mode='lines', name='Actual', line=dict(color='green')
))

# Plot predicted full series
fig.add_trace(go.Scatter(
    x=full_dates, y=full_pred,
    mode='lines', name='Predicted', line=dict(color='red', dash='dot')
))

fig.update_layout(
    title="Actual vs Predicted Values (Train + Test)",
    xaxis_title="Index",
    yaxis_title="Target Variable",
    template="plotly_white"
)

fig.show()
